# Demand Prediction Pipeline
**Goal**: Maximize R² score using ensemble of gradient boosting models

**Strategy**:
1. Feature engineering (timestamp, geohash, interactions)
2. Handle missing values natively (GBDT models)
3. Handle high-cardinality categoricals via label + target encoding
4. Ensemble: LightGBM + XGBoost + CatBoost with optimal blending

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully!')

ModuleNotFoundError: No module named 'lightgbm'

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

# NOTE: Run !pip install pygeohash in your terminal/notebook if you haven't yet
import pygeohash as pgh

# Load datasets
train = pd.read_csv('./dataset/train.csv')
test = pd.read_csv('./dataset/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'\nMissing values in train:')
print(train.isnull().sum()[train.isnull().sum() > 0])
train.head()

Train shape: (77299, 11)
Test shape:  (41778, 10)

Missing values in train:
RoadType        600
Temperature    2495
Weather         797
dtype: int64


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


## 2. Feature Engineering

In [2]:
target = 'demand'
y_train = train[target].values
train_idx = train['Index'].values
test_idx = test['Index'].values

# Track splits for safe combination processing
train['is_train'] = 1
test['is_train'] = 0
test['demand'] = np.nan

# Concatenate temporarily to ensure continuous spatial-temporal imputation
df = pd.concat([train, test], axis=0).reset_index(drop=True)

print("1/4. Decoding spatial coordinates...")
df['latitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[0])
df['longitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[1])

print("2/4. Decomposing temporal loops...")
parts = df['timestamp'].str.split(':', expand=True).astype(int)
df['hour'] = parts[0]
df['minute'] = parts[1]
df['time_minutes'] = df['hour'] * 60 + df['minute']

# Cyclical encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['min_sin'] = np.sin(2 * np.pi * df['time_minutes'] / 1440)
df['min_cos'] = np.cos(2 * np.pi * df['time_minutes'] / 1440)

# Time buckets
df['time_bucket'] = pd.cut(df['hour'], bins=[-1,5,9,12,17,21,24], labels=[0,1,2,3,4,5]).astype(int)

# Structural string modifications
df['geo_prefix4'] = df['geohash'].str[:4]
df['geo_prefix5'] = df['geohash'].str[:5]
df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
df['Landmarks_enc'] = (df['Landmarks'] == 'Yes').astype(int)

# Numeric mappings
df['RoadType_enc'] = df['RoadType'].map({'Residential': 0, 'Street': 1, 'Highway': 2})
df['Weather_enc'] = df['Weather'].map({'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3})
df['Temperature_missing'] = df['Temperature'].isnull().astype(int)

print("3/4. Executing spatial-temporal imputation...")
# Sort sequentially by timeline and location so forward-filling is physically accurate
df = df.sort_values(by=['geohash', 'day', 'time_minutes']).reset_index(drop=True)
df['Temperature'] = df.groupby('geohash')['Temperature'].ffill().bfill()
df['Weather_enc'] = df.groupby('geohash')['Weather_enc'].ffill().bfill()

# Safe road type mapping based on localized modes
global_road_mode = df['RoadType_enc'].mode()[0]
df['RoadType_enc'] = df.groupby('geohash')['RoadType_enc'].transform(lambda x: x.ffill().bfill().fillna(global_road_mode))

# Absolute fallback if any edge cases remain isolated
df['Temperature'] = df['Temperature'].fillna(df['Temperature'].mean())
df['Weather_enc'] = df['Weather_enc'].fillna(df['Weather_enc'].mode()[0])

print("4/4. Generating math interaction variables...")
df['lanes_x_road'] = df['NumberofLanes'] * df['RoadType_enc']
df['temp_x_weather'] = df['Temperature'] * df['Weather_enc']
df['lanes_x_landmarks'] = df['NumberofLanes'] * df['Landmarks_enc']
df['lanes_x_largeveh'] = df['NumberofLanes'] * df['LargeVehicles_enc']

print('Feature engineering base processing done!')

1/4. Decoding spatial coordinates...
2/4. Decomposing temporal loops...
3/4. Executing spatial-temporal imputation...
4/4. Generating math interaction variables...
Feature engineering base processing done!


In [3]:
print("1/2. Re-labeling structural strings...")
for col in ['geohash', 'geo_prefix4', 'geo_prefix5']:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# Recover clean train/test dataframes
train_fe = df[df['is_train'] == 1].copy()
test_fe = df[df['is_train'] == 0].copy()

print("2/2. Initializing leakage-free Out-of-Fold target metrics...")
train_fe['geo_target_mean'] = np.nan
train_fe['geo_target_std'] = np.nan
train_fe['geo_target_count'] = np.nan

# Replicating your friend's 5-Fold cross validation split strategy to calculate values
kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_mean = train_fe['demand'].mean()
global_std = train_fe['demand'].std()

for train_idx_fold, val_idx_fold in kf.split(train_fe):
    X_tr_fold = train_fe.iloc[train_idx_fold]
    
    # Calculate target statistics completely isolated within training fold
    geo_mean = X_tr_fold.groupby('geohash')['demand'].mean()
    geo_std = X_tr_fold.groupby('geohash')['demand'].std()
    geo_count = X_tr_fold.groupby('geohash')['demand'].count()
    
    # Safely apply to validation fold rows
    val_geohashes = train_fe.iloc[val_idx_fold]['geohash']
    train_fe.iloc[val_idx_fold, train_fe.columns.get_loc('geo_target_mean')] = val_geohashes.map(geo_mean)
    train_fe.iloc[val_idx_fold, train_fe.columns.get_loc('geo_target_std')] = val_geohashes.map(geo_std)
    train_fe.iloc[val_idx_fold, train_fe.columns.get_loc('geo_target_count')] = val_geohashes.map(geo_count)

# Fill unaligned categorical folds with global baselines
train_fe['geo_target_mean'] = train_fe['geo_target_mean'].fillna(global_mean)
train_fe['geo_target_std'] = train_fe['geo_target_std'].fillna(global_std)
train_fe['geo_target_count'] = train_fe['geo_target_count'].fillna(0)

# Test mappings rely on the aggregated structural metrics of the entire training set
full_geo_mean = train_fe.groupby('geohash')['demand'].mean()
full_geo_std = train_fe.groupby('geohash')['demand'].std()
full_geo_count = train_fe.groupby('geohash')['demand'].count()

test_fe['geo_target_mean'] = test_fe['geohash'].map(full_geo_mean).fillna(global_mean)
test_fe['geo_target_std'] = test_fe['geohash'].map(full_geo_std).fillna(global_std)
test_fe['geo_target_count'] = test_fe['geohash'].map(full_geo_count).fillna(0)

print('All encodings built safely without any target data leakage!')

1/2. Re-labeling structural strings...
2/2. Initializing leakage-free Out-of-Fold target metrics...
All encodings built safely without any target data leakage!


In [4]:
features = [
    'day', 'hour', 'minute', 'time_minutes',
    'hour_sin', 'hour_cos', 'min_sin', 'min_cos', 'time_bucket',
    'latitude', 'longitude',  # Fixed: Solves cold-start locations completely
    'geohash_enc', 'geo_prefix4_enc', 'geo_prefix5_enc',
    'geo_target_mean', 'geo_target_std', 'geo_target_count',
    'RoadType_enc', 'NumberofLanes', 'LargeVehicles_enc', 'Landmarks_enc',
    'Temperature', 'Temperature_missing', 'Weather_enc',
    'lanes_x_road', 'temp_x_weather', 'lanes_x_landmarks', 'lanes_x_largeveh',
]

X_train = train_fe[features].values
X_test = test_fe[features].values

print(f'Total features generated: {len(features)}')
print("\nReady for modeling! Send over the training and ensemble loop blocks next.")

Total features generated: 28

Ready for modeling! Send over the training and ensemble loop blocks next.


In [5]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 3. Model Training (5-Fold CV)
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

print("="*60)
print("TRAINING MODEL 1: LightGBM")
print("="*60)

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 63,          # Reduced from 127 to prevent overfitting the localized data
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.7,   # Lowered slightly to force more diverse tree creation
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.5,          # Increased L1 regularization
    'reg_lambda': 1.0,
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42,
}

oof_lgb = np.zeros(len(X_train))
pred_lgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(lgb_params, dtrain, num_boost_round=3000,
                      valid_sets=[dval],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
    
    oof_lgb[val_idx] = model.predict(X_val)
    pred_lgb += model.predict(X_test) / N_FOLDS
    print(f'LGB Fold {fold+1} R2: {r2_score(y_val, oof_lgb[val_idx]):.6f}')

print(f'>>> LightGBM Overall OOF R2: {r2_score(y_train, oof_lgb):.6f}\n')

TRAINING MODEL 1: LightGBM
LGB Fold 1 R2: 0.053982
LGB Fold 2 R2: 0.048573
LGB Fold 3 R2: 0.044456
LGB Fold 4 R2: 0.052195
LGB Fold 5 R2: 0.046285
>>> LightGBM Overall OOF R2: 0.049082



In [6]:
print("="*60)
print("TRAINING MODEL 2: XGBoost")
print("="*60)

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.03,
    'max_depth': 8,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 2.0,         # High L2 penalty to keep predictions grounded
    'tree_method': 'hist',
    'random_state': 42,
}

oof_xgb = np.zeros(len(X_train))
pred_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=features)
    dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
    dtest = xgb.DMatrix(X_test, feature_names=features)
    
    model = xgb.train(xgb_params, dtrain, num_boost_round=3000,
                      evals=[(dval, 'val')], early_stopping_rounds=100, verbose_eval=False)
    
    oof_xgb[val_idx] = model.predict(dval)
    pred_xgb += model.predict(dtest) / N_FOLDS
    print(f'XGB Fold {fold+1} R2: {r2_score(y_val, oof_xgb[val_idx]):.6f}')

print(f'>>> XGBoost Overall OOF R2: {r2_score(y_train, oof_xgb):.6f}\n')

TRAINING MODEL 2: XGBoost
XGB Fold 1 R2: 0.051813
XGB Fold 2 R2: 0.051210
XGB Fold 3 R2: 0.046454
XGB Fold 4 R2: 0.052963
XGB Fold 5 R2: 0.045956
>>> XGBoost Overall OOF R2: 0.049674



In [7]:
print("="*60)
print("TRAINING MODEL 3: CatBoost")
print("="*60)

oof_cat = np.zeros(len(X_train))
pred_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    # Using symmetric trees to perfectly offset LightGBM's asymmetry
    model = CatBoostRegressor(
        iterations=3000, 
        learning_rate=0.03, 
        depth=8,
        l2_leaf_reg=5, 
        random_seed=42, 
        verbose=0,
        early_stopping_rounds=100, 
        eval_metric='RMSE',
        bootstrap_type='Bernoulli',
        subsample=0.8
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
    
    oof_cat[val_idx] = model.predict(X_val)
    pred_cat += model.predict(X_test) / N_FOLDS
    print(f'CAT Fold {fold+1} R2: {r2_score(y_val, oof_cat[val_idx]):.6f}')

print(f'>>> CatBoost Overall OOF R2: {r2_score(y_train, oof_cat):.6f}\n')

TRAINING MODEL 3: CatBoost
CAT Fold 1 R2: 0.043763
CAT Fold 2 R2: 0.040833
CAT Fold 3 R2: 0.039156
CAT Fold 4 R2: 0.043789
CAT Fold 5 R2: 0.041462
>>> CatBoost Overall OOF R2: 0.041798



In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pygeohash as pgh

# 1. Reload raw data fresh to reset state
train = pd.read_csv('./dataset/train.csv')
test = pd.read_csv('./dataset/test.csv')

# Save target index mapping BEFORE any sorting happens
train_idx = train['Index'].values
test_idx = test['Index'].values

train['is_train'] = 1
test['is_train'] = 0
test['demand'] = np.nan

# Combine datasets
df = pd.concat([train, test], axis=0).reset_index(drop=True)

# 2. Extract Features (WITHOUT global dataframe sorting to preserve raw row index alignment)
print("Processing spatial-temporal layout...")
df['latitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[0])
df['longitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[1])

parts = df['timestamp'].str.split(':', expand=True).astype(int)
df['hour'] = parts[0]
df['minute'] = parts[1]
df['time_minutes'] = df['hour'] * 60 + df['minute']

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['min_sin'] = np.sin(2 * np.pi * df['time_minutes'] / 1440)
df['min_cos'] = np.cos(2 * np.pi * df['time_minutes'] / 1440)

df['time_bucket'] = pd.cut(df['hour'], bins=[-1,5,9,12,17,21,24], labels=[0,1,2,3,4,5]).astype(int)
df['geo_prefix4'] = df['geohash'].str[:4]
df['geo_prefix5'] = df['geohash'].str[:5]
df['LargeVehicles_enc'] = (df['LargeVehicles'] == 'Allowed').astype(int)
df['Landmarks_enc'] = (df['Landmarks'] == 'Yes').astype(int)

df['RoadType_enc'] = df['RoadType'].map({'Residential': 0, 'Street': 1, 'Highway': 2})
df['Weather_enc'] = df['Weather'].map({'Sunny': 0, 'Rainy': 1, 'Foggy': 2, 'Snowy': 3})
df['Temperature_missing'] = df['Temperature'].isnull().astype(int)

# 3. Safe Imputation via Mapping (No sorting required!)
print("Imputing values safely...")
# Calculate median values per geohash to map back safely without shuffling rows
geo_temp_map = df.groupby('geohash')['Temperature'].transform('median')
df['Temperature'] = df['Temperature'].fillna(geo_temp_map).fillna(df['Temperature'].median())

geo_weather_map = df.groupby('geohash')['Weather_enc'].transform(lambda x: x.mode()[0] if not x.mode().empty else 0)
df['Weather_enc'] = df['Weather_enc'].fillna(geo_weather_map).fillna(0)

geo_road_map = df.groupby('geohash')['RoadType_enc'].transform(lambda x: x.mode()[0] if not x.mode().empty else 0)
df['RoadType_enc'] = df['RoadType_enc'].fillna(geo_road_map).fillna(0)

# Interactions
df['lanes_x_road'] = df['NumberofLanes'] * df['RoadType_enc']
df['temp_x_weather'] = df['Temperature'] * df['Weather_enc']
df['lanes_x_landmarks'] = df['NumberofLanes'] * df['Landmarks_enc']
df['lanes_x_largeveh'] = df['NumberofLanes'] * df['LargeVehicles_enc']

# Label Encodings
for col in ['geohash', 'geo_prefix4', 'geo_prefix5']:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

# 4. Separate sets cleanly preserving exact row structure
train_fe = df[df['is_train'] == 1].copy()
test_fe = df[df['is_train'] == 0].copy()

# Pull target variable directly from aligned dataframe
y_train = train_fe['demand'].values

# Out-of-fold target encoding cleanly aligned to rows
from sklearn.model_selection import KFold
train_fe['geo_target_mean'] = np.nan
train_fe['geo_target_std'] = np.nan
train_fe['geo_target_count'] = np.nan

kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_mean = y_train.mean()
global_std = y_train.std()

for tr_idx_f, val_idx_f in kf.split(train_fe):
    X_tr_f = train_fe.iloc[tr_idx_f]
    geo_mean = X_tr_f.groupby('geohash')['demand'].mean()
    geo_std = X_tr_f.groupby('geohash')['demand'].std()
    geo_count = X_tr_f.groupby('geohash')['demand'].count()
    
    val_geos = train_fe.iloc[val_idx_f]['geohash']
    train_fe.iloc[val_idx_f, train_fe.columns.get_loc('geo_target_mean')] = val_geos.map(geo_mean)
    train_fe.iloc[val_idx_f, train_fe.columns.get_loc('geo_target_std')] = val_geos.map(geo_std)
    train_fe.iloc[val_idx_f, train_fe.columns.get_loc('geo_target_count')] = val_geos.map(geo_count)

train_fe['geo_target_mean'] = train_fe['geo_target_mean'].fillna(global_mean)
train_fe['geo_target_std'] = train_fe['geo_target_std'].fillna(global_std)
train_fe['geo_target_count'] = train_fe['geo_target_count'].fillna(0)

# Test maps
full_geo_mean = train_fe.groupby('geohash')['demand'].mean()
full_geo_std = train_fe.groupby('geohash')['demand'].std()
full_geo_count = train_fe.groupby('geohash')['demand'].count()

test_fe['geo_target_mean'] = test_fe['geohash'].map(full_geo_mean).fillna(global_mean)
test_fe['geo_target_std'] = test_fe['geohash'].map(full_geo_std).fillna(global_std)
test_fe['geo_target_count'] = test_fe['geohash'].map(full_geo_count).fillna(0)

# Build Matrices
features = [
    'day', 'hour', 'minute', 'time_minutes',
    'hour_sin', 'hour_cos', 'min_sin', 'min_cos', 'time_bucket',
    'latitude', 'longitude',
    'geohash_enc', 'geo_prefix4_enc', 'geo_prefix5_enc',
    'geo_target_mean', 'geo_target_std', 'geo_target_count',
    'RoadType_enc', 'NumberofLanes', 'LargeVehicles_enc', 'Landmarks_enc',
    'Temperature', 'Temperature_missing', 'Weather_enc',
    'lanes_x_road', 'temp_x_weather', 'lanes_x_landmarks', 'lanes_x_largeveh',
]

X_train = train_fe[features].values
X_test = test_fe[features].values

print(f"Features: {X_train.shape[1]}, Target rows: {len(y_train)}")
print("Row order verified. Ready to re-run the modeling loops!")

Processing spatial-temporal layout...
Imputing values safely...
Features: 28, Target rows: 77299
Row order verified. Ready to re-run the modeling loops!


In [10]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 1. Initialize GroupKFold to handle spatial generalization
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

# Define groups using the spatial geohash strings from the processed training dataframe
groups = train_fe['geohash'].values

# Initialize arrays for Out-of-Fold (OOF) and Test predictions
oof_lgb, pred_lgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_xgb, pred_xgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_cat, pred_cat = np.zeros(len(X_train)), np.zeros(len(X_test))

# Apply log transformation to the target variable to stabilize training variance
y_train_log = np.log1p(y_train)

print("="*60)
print("TRAINING ENSEMBLE MODELS WITH SPATIAL HOLDOUT (GROUP-KFOLD)")
print("="*60)

# Loop through spatial-split folds instead of shuffled rows
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train_log, groups)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr_log, y_val_log = y_train_log[tr_idx], y_train_log[val_idx]
    y_val_true = y_train[val_idx] # Tracking on the original scale
    
    # --- 1. LIGHTGBM (Lower LR, forced regularization to generalize across map) ---
    lgb_params = {
        'objective': 'regression', 
        'metric': 'rmse', 
        'boosting_type': 'gbdt',
        'learning_rate': 0.015,       # Slower learning rate for deep spatial trees
        'num_leaves': 63,             # Reduced to avoid overfitting regional quirks
        'max_depth': -1,
        'min_child_samples': 40, 
        'feature_fraction': 0.6,      # Forces tree variety per split
        'bagging_fraction': 0.7, 
        'bagging_freq': 3, 
        'reg_alpha': 0.5, 
        'reg_lambda': 2.0, 
        'verbose': -1, 
        'n_jobs': -1, 
        'random_state': 42
    }
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr_log)
    dval_lgb = lgb.Dataset(X_val, label=y_val_log)
    model_lgb = lgb.train(lgb_params, dtrain_lgb, num_boost_round=4000, 
                          valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(150, verbose=False)])
    
    oof_lgb[val_idx] = np.expm1(model_lgb.predict(X_val))
    pred_lgb += np.expm1(model_lgb.predict(X_test)) / N_FOLDS
    
    # --- 2. XGBOOST (Conservative Spatial Anchor) ---
    xgb_params = {
        'objective': 'reg:squarederror', 
        'eval_metric': 'rmse', 
        'learning_rate': 0.015,
        'max_depth': 6, 
        'min_child_weight': 15, 
        'subsample': 0.7, 
        'colsample_bytree': 0.6,
        'reg_alpha': 0.5, 
        'reg_lambda': 3.0, 
        'tree_method': 'hist', 
        'random_state': 42
    }
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr_log, feature_names=features)
    dval_xgb = xgb.DMatrix(X_val, label=y_val_log, feature_names=features)
    dtest_xgb = xgb.DMatrix(X_test, feature_names=features)
    model_xgb = xgb.train(xgb_params, dtrain_xgb, num_boost_round=4000, 
                          evals=[(dval_xgb, 'val')], early_stopping_rounds=150, verbose_eval=False)
    
    oof_xgb[val_idx] = np.expm1(model_xgb.predict(dval_xgb))
    pred_xgb += np.expm1(model_xgb.predict(dtest_xgb)) / N_FOLDS
    
    # --- 3. CATBOOST (Robust handling of engineered structural categoricals) ---
    model_cat = CatBoostRegressor(
        iterations=4000, 
        learning_rate=0.02, 
        depth=7, 
        l2_leaf_reg=5, 
        random_seed=42, 
        verbose=0, 
        early_stopping_rounds=150, 
        eval_metric='RMSE'
    )
    model_cat.fit(X_tr, y_tr_log, eval_set=(X_val, y_val_log), verbose=0)
    
    oof_cat[val_idx] = np.expm1(model_cat.predict(X_val))
    pred_cat += np.expm1(model_cat.predict(X_test)) / N_FOLDS
    
    # Evaluate current fold performance on original scale
    fold_blend = (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3.0
    print(f"Fold {fold+1} Spatial Holdout Blend R2: {r2_score(y_val_true, fold_blend):.6f}")

print("\n" + "="*60)
print("MATHEMATICAL WEIGHT OPTIMIZATION (SCIPY SLSQP)")
print("="*60)

# Objective: Minimize negative R2 to extract the optimal mathematical blend weight
def r2_objective(weights):
    blend = weights[0] * oof_lgb + weights[1] * oof_xgb + weights[2] * oof_cat
    return -r2_score(y_train, blend)

cons = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
bounds = [(0, 1), (0, 1), (0, 1)]
opt_res = minimize(r2_objective, [0.34, 0.33, 0.33], method='SLSQP', bounds=bounds, constraints=cons)
best_w = opt_res.x

print(f"Optimal Weights -> LGB: {best_w[0]:.4f} | XGB: {best_w[1]:.4f} | CAT: {best_w[2]:.4f}")
print(f"Individual OOF R2 Scores:")
print(f"  LightGBM R2: {r2_score(y_train, oof_lgb):.6f}")
print(f"  XGBoost R2:  {r2_score(y_train, oof_xgb):.6f}")
print(f"  CatBoost R2: {r2_score(y_train, oof_cat):.6f}")
print(f"👉 MAXIMIZED ENSEMBLE OOF R2: {-opt_res.fun:.6f}\n")

# Process and export submission file
final_pred = best_w[0] * pred_lgb + best_w[1] * pred_xgb + best_w[2] * pred_cat
submission = pd.DataFrame({'Index': test_idx, 'demand': final_pred})
submission.to_csv('./predicted_demand_optimized.csv', index=False)
print(f"Saved: predicted_demand_optimized.csv ({submission.shape[0]} rows)")

TRAINING ENSEMBLE MODELS WITH SPATIAL HOLDOUT (GROUP-KFOLD)
Fold 1 Spatial Holdout Blend R2: 0.919141
Fold 2 Spatial Holdout Blend R2: 0.916132
Fold 3 Spatial Holdout Blend R2: 0.929006
Fold 4 Spatial Holdout Blend R2: 0.915534
Fold 5 Spatial Holdout Blend R2: 0.940272

MATHEMATICAL WEIGHT OPTIMIZATION (SCIPY SLSQP)
Optimal Weights -> LGB: 0.3632 | XGB: 0.0000 | CAT: 0.6368
Individual OOF R2 Scores:
  LightGBM R2: 0.923948
  XGBoost R2:  0.922241
  CatBoost R2: 0.924778
👉 MAXIMIZED ENSEMBLE OOF R2: 0.925820

Saved: predicted_demand_optimized.csv (41778 rows)


In [11]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# =====================================================================
# 🌟 THE GOLDEN FEATURE: Spatial-Temporal Target Encoding
# We create a baseline demand for specific locations at specific times
# =====================================================================
print("Engineeering Spatial-Temporal Baseline...")
# Assuming 'time_bucket' (0-5 for times of day) and 'geohash' exist in train_fe and test_fe
geo_time_mean = train_fe.groupby(['geohash', 'time_bucket'])['demand'].mean().reset_index()
geo_time_mean.rename(columns={'demand': 'geo_time_baseline'}, inplace=True)

# Merge back into train and test
train_fe = train_fe.merge(geo_time_mean, on=['geohash', 'time_bucket'], how='left')
test_fe = test_fe.merge(geo_time_mean, on=['geohash', 'time_bucket'], how='left')

# Fill any rare missing combinations with the global average
global_mean = train_fe['demand'].mean()
train_fe['geo_time_baseline'] = train_fe['geo_time_baseline'].fillna(global_mean)
test_fe['geo_time_baseline'] = test_fe['geo_time_baseline'].fillna(global_mean)

# Re-build X_train and X_test with the new feature
features_updated = features + ['geo_time_baseline']
X_train = train_fe[features_updated].values
X_test = test_fe[features_updated].values
# =====================================================================

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb, pred_lgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_xgb, pred_xgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_cat, pred_cat = np.zeros(len(X_train)), np.zeros(len(X_test))

# Log transform is kept to handle massive demand spikes cleanly
y_train_log = np.log1p(y_train)

print("="*60)
print("TRAINING ENSEMBLE WITH SPATIAL-TEMPORAL MEMORIZATION")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr_log, y_val_log = y_train_log[tr_idx], y_train_log[val_idx]
    y_val_true = y_train[val_idx] 
    
    # --- 1. LIGHTGBM ---
    lgb_params = {
        'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
        'learning_rate': 0.03, 'num_leaves': 127, 'max_depth': -1,
        'min_child_samples': 20, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
        'bagging_freq': 5, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'verbose': -1, 'n_jobs': -1, 'random_state': 42
    }
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr_log)
    dval_lgb = lgb.Dataset(X_val, label=y_val_log)
    model_lgb = lgb.train(lgb_params, dtrain_lgb, num_boost_round=3000, valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(100, verbose=False)])
    
    oof_lgb[val_idx] = np.expm1(model_lgb.predict(X_val))
    pred_lgb += np.expm1(model_lgb.predict(X_test)) / N_FOLDS
    
    # --- 2. XGBOOST (Untethered to ensure it captures unique logic) ---
    xgb_params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03,
        'max_depth': 8, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'reg_alpha': 0.1, 'reg_lambda': 1.0, 'tree_method': 'hist', 'random_state': 42
    }
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr_log, feature_names=features_updated)
    dval_xgb = xgb.DMatrix(X_val, label=y_val_log, feature_names=features_updated)
    dtest_xgb = xgb.DMatrix(X_test, feature_names=features_updated)
    model_xgb = xgb.train(xgb_params, dtrain_xgb, num_boost_round=3000, evals=[(dval_xgb, 'val')], early_stopping_rounds=100, verbose_eval=False)
    
    oof_xgb[val_idx] = np.expm1(model_xgb.predict(dval_xgb))
    pred_xgb += np.expm1(model_xgb.predict(dtest_xgb)) / N_FOLDS
    
    # --- 3. CATBOOST ---
    model_cat = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=8, l2_leaf_reg=3, random_seed=42, verbose=0, early_stopping_rounds=100, eval_metric='RMSE')
    model_cat.fit(X_tr, y_tr_log, eval_set=(X_val, y_val_log), verbose=0)
    
    oof_cat[val_idx] = np.expm1(model_cat.predict(X_val))
    pred_cat += np.expm1(model_cat.predict(X_test)) / N_FOLDS
    
    fold_blend = (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3.0
    print(f"Fold {fold+1} R2: {r2_score(y_val_true, fold_blend):.6f}")

print("\n" + "="*60)
print("MATHEMATICAL WEIGHT OPTIMIZATION (SCIPY SLSQP)")
print("="*60)

def r2_objective(weights):
    blend = weights[0] * oof_lgb + weights[1] * oof_xgb + weights[2] * oof_cat
    return -r2_score(y_train, blend)

cons = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
bounds = [(0, 1), (0, 1), (0, 1)]
opt_res = minimize(r2_objective, [0.34, 0.33, 0.33], method='SLSQP', bounds=bounds, constraints=cons)
best_w = opt_res.x

print(f"Optimal Weights -> LGB: {best_w[0]:.4f} | XGB: {best_w[1]:.4f} | CAT: {best_w[2]:.4f}")
print(f"👉 MAXIMIZED ENSEMBLE OOF R2: {-opt_res.fun:.6f}\n")

# Process submission file
final_pred = best_w[0] * pred_lgb + best_w[1] * pred_xgb + best_w[2] * pred_cat

# Post-processing: Demand can't be negative. Clip at 0 just in case.
final_pred = np.clip(final_pred, 0, None)

submission = pd.DataFrame({'Index': test_idx, 'demand': final_pred})
submission.to_csv('./predicted_demand_optimized_v3.csv', index=False)
print(f"Saved: predicted_demand_optimized_v3.csv ({submission.shape[0]} rows)")

Engineeering Spatial-Temporal Baseline...
TRAINING ENSEMBLE WITH SPATIAL-TEMPORAL MEMORIZATION
Fold 1 R2: 0.964206
Fold 2 R2: 0.964725
Fold 3 R2: 0.966012
Fold 4 R2: 0.958766
Fold 5 R2: 0.965396

MATHEMATICAL WEIGHT OPTIMIZATION (SCIPY SLSQP)
Optimal Weights -> LGB: 0.3400 | XGB: 0.3300 | CAT: 0.3300
👉 MAXIMIZED ENSEMBLE OOF R2: 0.963882

Saved: predicted_demand_optimized_v3.csv (41778 rows)


In [13]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# =====================================================================
# 1. BULLETPROOF TIME-SERIES LAG ENGINEERING
# =====================================================================
print("Engineering 24-hour Lag and Trend features...")

# Combine datasets while tracking original order to prevent row misalignment
all_data = pd.concat([train_fe, test_fe], axis=0).reset_index(drop=True)
all_data['orig_sort_idx'] = range(len(all_data))

# Create spatial-temporal tracking key
all_data['time_key'] = all_data['geohash'].astype(str) + "_" + all_data['time_minutes'].astype(str)

# Map demand from Day 48 -> Day 49, and Day 49 -> Day 50
lag_map = all_data[all_data['day'].isin([48, 49])][['time_key', 'day', 'demand']].copy()
lag_map['target_day'] = lag_map['day'] + 1
lag_map = lag_map.rename(columns={'demand': 'demand_lag_24h'}).drop('day', axis=1)

# Merge lag features back cleanly
all_data = all_data.merge(lag_map, left_on=['time_key', 'day'], right_on=['time_key', 'target_day'], how='left')

# Backfill historical gaps using robust regional medians
all_data['demand_lag_24h'] = all_data['demand_lag_24h'].fillna(all_data.groupby('geohash')['demand'].transform('median'))
all_data['demand_lag_24h'] = all_data['demand_lag_24h'].fillna(all_data['demand'].median())

# Create historical baseline delta tracking
all_data['demand_trend'] = all_data['demand_lag_24h'] - all_data.groupby('geohash')['demand_lag_24h'].shift(1).fillna(0)

# Restore exact original row sequencing
all_data = all_data.sort_values('orig_sort_idx').reset_index(drop=True)

# Split back to independent feature sets
train_fe_updated = all_data[all_data['is_train'] == 1].copy()
test_fe_updated = all_data[all_data['is_train'] == 0].copy()

# Update feature matrix lists
features_updated = features + ['demand_lag_24h', 'demand_trend']

X_train = train_fe_updated[features_updated].values
X_test = test_fe_updated[features_updated].values
y_train = train_fe_updated['demand'].values

# =====================================================================
# 2. 5-FOLD ENSEMBLE TRAINING ENGINE (WITH TARGET LOG TRANSFORMATION)
# =====================================================================
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb, pred_lgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_xgb, pred_xgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_cat, pred_cat = np.zeros(len(X_train)), np.zeros(len(X_test))

y_train_log = np.log1p(y_train)

print("="*60)
print("TRAINING SUPERCHARGED SPATIAL-TEMPORAL ENSEMBLE")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr_log, y_val_log = y_train_log[tr_idx], y_train_log[val_idx]
    y_val_true = y_train[val_idx]
    
    # --- 1. LIGHTGBM ---
    lgb_params = {
        'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
        'learning_rate': 0.02, 'num_leaves': 127, 'max_depth': -1,
        'min_child_samples': 30, 'feature_fraction': 0.7, 'bagging_fraction': 0.8,
        'bagging_freq': 5, 'reg_alpha': 0.2, 'reg_lambda': 1.5, 'verbose': -1, 'n_jobs': -1, 'random_state': 42
    }
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr_log)
    dval_lgb = lgb.Dataset(X_val, label=y_val_log)
    model_lgb = lgb.train(lgb_params, dtrain_lgb, num_boost_round=3500, valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(120, verbose=False)])
    
    oof_lgb[val_idx] = np.expm1(model_lgb.predict(X_val))
    pred_lgb += np.expm1(model_lgb.predict(X_test)) / N_FOLDS
    
    # --- 2. XGBOOST ---
    xgb_params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.02,
        'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.8, 'colsample_bytree': 0.7,
        'reg_alpha': 0.2, 'reg_lambda': 2.0, 'tree_method': 'hist', 'random_state': 42
    }
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr_log, feature_names=features_updated)
    dval_xgb = xgb.DMatrix(X_val, label=y_val_log, feature_names=features_updated)
    dtest_xgb = xgb.DMatrix(X_test, feature_names=features_updated)
    model_xgb = xgb.train(xgb_params, dtrain_xgb, num_boost_round=3500, evals=[(dval_xgb, 'val')], early_stopping_rounds=120, verbose_eval=False)
    
    oof_xgb[val_idx] = np.expm1(model_xgb.predict(dval_xgb))
    pred_xgb += np.expm1(model_xgb.predict(dtest_xgb)) / N_FOLDS
    
    # --- 3. CATBOOST ---
    model_cat = CatBoostRegressor(iterations=3500, learning_rate=0.025, depth=7, l2_leaf_reg=4, random_seed=42, verbose=0, early_stopping_rounds=120, eval_metric='RMSE')
    model_cat.fit(X_tr, y_tr_log, eval_set=(X_val, y_val_log), verbose=0)
    
    oof_cat[val_idx] = np.expm1(model_cat.predict(X_val))
    pred_cat += np.expm1(model_cat.predict(X_test)) / N_FOLDS
    
    fold_blend = (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3.0
    print(f"Fold {fold+1} Blended R2: {r2_score(y_val_true, fold_blend):.6f}")

# =====================================================================
# 3. MATHEMATICAL WEIGHT OPTIMIZATION (SCIPY SLSQP)
# =====================================================================
print("\n" + "="*60)
print("OPTIMIZING FINAL ENSEMBLE BLEND MATRIX")
print("="*60)

def r2_objective(weights):
    blend = weights[0] * oof_lgb + weights[1] * oof_xgb + weights[2] * oof_cat
    return -r2_score(y_train, blend)

cons = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
bounds = [(0, 1), (0, 1), (0, 1)]
opt_res = minimize(r2_objective, [0.34, 0.33, 0.33], method='SLSQP', bounds=bounds, constraints=cons)
best_w = opt_res.x

print(f"Optimal Weights -> LGB: {best_w[0]:.4f} | XGB: {best_w[1]:.4f} | CAT: {best_w[2]:.4f}")
print(f"Individual OOF R2 Scores:")
print(f"  LightGBM R2: {r2_score(y_train, oof_lgb):.6f}")
print(f"  XGBoost R2:  {r2_score(y_train, oof_xgb):.6f}")
print(f"  CatBoost R2: {r2_score(y_train, oof_cat):.6f}")
print(f"👉 MAXIMIZED ENSEMBLE OOF R2: {-opt_res.fun:.6f}\n")

# =====================================================================
# 4. POST-PROCESSING AND SUBMISSION GENERATION
# =====================================================================
final_pred = best_w[0] * pred_lgb + best_w[1] * pred_xgb + best_w[2] * pred_cat

# Standard guardrail boundary adjustment
final_pred = np.clip(final_pred, 0, None)

submission = pd.DataFrame({'Index': test_idx, 'demand': final_pred})
submission.to_csv('./predicted_demand_final_ensemble.csv', index=False)
print(f"Saved: predicted_demand_final_ensemble.csv ({submission.shape[0]} rows)")

Engineering 24-hour Lag and Trend features...
TRAINING SUPERCHARGED SPATIAL-TEMPORAL ENSEMBLE
Fold 1 Blended R2: 0.956362
Fold 2 Blended R2: 0.955986
Fold 3 Blended R2: 0.958113
Fold 4 Blended R2: 0.951688
Fold 5 Blended R2: 0.957058

OPTIMIZING FINAL ENSEMBLE BLEND MATRIX
Optimal Weights -> LGB: 0.3400 | XGB: 0.3300 | CAT: 0.3300
Individual OOF R2 Scores:
  LightGBM R2: 0.954656
  XGBoost R2:  0.955478
  CatBoost R2: 0.954064
👉 MAXIMIZED ENSEMBLE OOF R2: 0.955888

Saved: predicted_demand_final_ensemble.csv (41778 rows)


In [15]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# =====================================================================
# 🛠️ UNIQUE ENGINE: PURE PYTHON GEOCENTRIC DECODER (RANK 1)
# =====================================================================
def geohash_to_latlon(geohash):
    """Decodes standard base32 geohashes into float Latitude/Longitude centers"""
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    for char in geohash:
        bit_value = base32.find(char)
        if bit_value == -1: 
            continue
        for i in range(4, -1, -1):
            bit = (bit_value >> i) & 1
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2
                if bit == 1: lon_interval = (mid, lon_interval[1])
                else: lon_interval = (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2
                if bit == 1: lat_interval = (mid, lat_interval[1])
                else: lat_interval = (lat_interval[0], mid)
            is_even = not is_even
    return (lat_interval[0] + lat_interval[1]) / 2, (lon_interval[0] + lon_interval[1]) / 2

print("Decomposing Spatial Geometry and Temporal Cycles...")
all_data = pd.concat([train_fe, test_fe], axis=0).reset_index(drop=True)

# Apply Spatial Decomposition safely
coords = [geohash_to_latlon(gh) for gh in all_data['geohash']]
all_data['latitude'] = [c[0] for c in coords]
all_data['longitude'] = [c[1] for c in coords]

# Apply Cyclic Time Encodings (Rank 2)
all_data['sin_time'] = np.sin(2 * np.pi * all_data['time_minutes'] / 1440.0)
all_data['cos_time'] = np.cos(2 * np.pi * all_data['time_minutes'] / 1440.0)

# Isolate structural components back to clean matrices
train_fe_updated = all_data[all_data['is_train'] == 1].copy()
test_fe_updated = all_data[all_data['is_train'] == 0].copy()

# ⚡ FIX: Build updated feature space completely removing any string duplicates
raw_features_v5 = [f for f in features if f != 'geohash'] + ['latitude', 'longitude', 'sin_time', 'cos_time']
features_v5 = list(dict.fromkeys(raw_features_v5))

X_train = train_fe_updated[features_v5].values
X_test = test_fe_updated[features_v5].values
y_train = train_fe_updated['demand'].values

# Handle any unexpected NaNs safely
X_train = np.nan_to_num(X_train)
X_test = np.nan_to_num(X_test)

# =====================================================================
# 🤖 4-MODEL CROSS-VALIDATION LOOP (RANK 3)
# =====================================================================
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb, pred_lgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_xgb, pred_xgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_cat, pred_cat = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_et,  pred_et  = np.zeros(len(X_train)), np.zeros(len(X_test))

# Keeping log-scale target for boosting stability across spatial groups
y_train_log = np.log1p(y_train)

print("="*60)
print("TRAINING GEOCENTRIC 4-MODEL HETEROGENEOUS ENSEMBLE")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr_log, y_val_log = y_train_log[tr_idx], y_train_log[val_idx]
    y_val_true = y_train[val_idx]
    
    # 1. LightGBM
    lgb_params = {
        'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
        'learning_rate': 0.03, 'num_leaves': 127, 'max_depth': -1,
        'min_child_samples': 20, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
        'bagging_freq': 5, 'verbose': -1, 'n_jobs': -1, 'random_state': 42
    }
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr_log)
    dval_lgb = lgb.Dataset(X_val, label=y_val_log)
    model_lgb = lgb.train(lgb_params, dtrain_lgb, num_boost_round=3000, valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[val_idx] = np.expm1(model_lgb.predict(X_val))
    pred_lgb += np.expm1(model_lgb.predict(X_test)) / N_FOLDS
    
    # 2. XGBoost
    xgb_params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.03,
        'max_depth': 8, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'tree_method': 'hist', 'random_state': 42
    }
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr_log, feature_names=features_v5)
    dval_xgb = xgb.DMatrix(X_val, label=y_val_log, feature_names=features_v5)
    dtest_xgb = xgb.DMatrix(X_test, feature_names=features_v5)
    model_xgb = xgb.train(xgb_params, dtrain_xgb, num_boost_round=3000, evals=[(dval_xgb, 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb[val_idx] = np.expm1(model_xgb.predict(dval_xgb))
    pred_xgb += np.expm1(model_xgb.predict(dtest_xgb)) / N_FOLDS
    
    # 3. CatBoost
    model_cat = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=8, random_seed=42, verbose=0, early_stopping_rounds=100, eval_metric='RMSE')
    model_cat.fit(X_tr, y_tr_log, eval_set=(X_val, y_val_log), verbose=0)
    oof_cat[val_idx] = np.expm1(model_cat.predict(X_val))
    pred_cat += np.expm1(model_cat.predict(X_test)) / N_FOLDS
    
    # 4. ExtraTrees (Variance Regulator trained directly on the raw demand scale)
    model_et = ExtraTreesRegressor(n_estimators=150, max_depth=20, min_samples_split=5, max_features=0.7, n_jobs=-1, random_state=42)
    model_et.fit(X_tr, y_train[tr_idx])
    oof_et[val_idx] = model_et.predict(X_val)
    pred_et += model_et.predict(X_test) / N_FOLDS
    
    fold_blend = (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx] + oof_et[val_idx]) / 4.0
    print(f"Fold {fold+1} Geocentric Blend R2: {r2_score(y_val_true, fold_blend):.6f}")

# =====================================================================
# 🎛️ 4-DIMENSIONAL MATH WEIGHT OPTIMIZATION
# =====================================================================
print("\n" + "="*60)
print("OPTIMIZING 4-WAY BALANCING MATRIX (SLSQP)")
print("="*60)

def r2_objective(weights):
    blend = weights[0]*oof_lgb + weights[1]*oof_xgb + weights[2]*oof_cat + weights[3]*oof_et
    return -r2_score(y_train, blend)

cons = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
bounds = [(0, 1)] * 4
opt_res = minimize(r2_objective, [0.25, 0.25, 0.25, 0.25], method='SLSQP', bounds=bounds, constraints=cons)
best_w = opt_res.x

print(f"Optimal Weights -> LGB: {best_w[0]:.4f} | XGB: {best_w[1]:.4f} | CAT: {best_w[2]:.4f} | ET: {best_w[3]:.4f}")
print(f"Individual OOF R2 Scores:")
print(f"  LightGBM R2:   {r2_score(y_train, oof_lgb):.6f}")
print(f"  XGBoost R2:    {r2_score(y_train, oof_xgb):.6f}")
print(f"  CatBoost R2:   {r2_score(y_train, oof_cat):.6f}")
print(f"  ExtraTrees R2: {r2_score(y_train, oof_et):.6f}")
print(f"👉 MAXIMIZED 4-MODEL ENSEMBLE OOF R2: {-opt_res.fun:.6f}\n")

# Process final predictions
final_pred = best_w[0]*pred_lgb + best_w[1]*pred_xgb + best_w[2]*pred_cat + best_w[3]*pred_et
final_pred = np.clip(final_pred, 0, None)

submission = pd.DataFrame({'Index': test_idx, 'demand': final_pred})
submission.to_csv('./predicted_demand_geocentric.csv', index=False)
print(f"Saved: predicted_demand_geocentric.csv ({submission.shape[0]} rows)")

Decomposing Spatial Geometry and Temporal Cycles...
TRAINING GEOCENTRIC 4-MODEL HETEROGENEOUS ENSEMBLE
Fold 1 Geocentric Blend R2: 0.957639
Fold 2 Geocentric Blend R2: 0.957209
Fold 3 Geocentric Blend R2: 0.958515
Fold 4 Geocentric Blend R2: 0.952557
Fold 5 Geocentric Blend R2: 0.956705

OPTIMIZING 4-WAY BALANCING MATRIX (SLSQP)
Optimal Weights -> LGB: 0.1080 | XGB: 0.1818 | CAT: 0.2290 | ET: 0.4812
Individual OOF R2 Scores:
  LightGBM R2:   0.953732
  XGBoost R2:    0.954533
  CatBoost R2:   0.954650
  ExtraTrees R2: 0.955493
👉 MAXIMIZED 4-MODEL ENSEMBLE OOF R2: 0.956890

Saved: predicted_demand_geocentric.csv (41778 rows)


In [ ]:
---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[16], line 49
     45 all_data['cos_time'] = np.cos(2 * np.pi * all_data['time_minutes'] / 1440.0)
     47 # 🌟 NEW FEATURE: Spatial Demand Volatility (Standard Deviation per Coordinate)
     48 # Tracks the behavioral risk/variance of different regions
---> 49 geo_stats = all_data[all_data['is_train'] == 1].groupby('geohash')['demand'].agg(['std', 'mad']).reset_index()
     50 geo_stats.rename(columns={'std': 'geo_demand_std', 'mad': 'geo_demand_mad'}, inplace=True)
     52 all_data = all_data.merge(geo_stats, on='geohash', how='left')

File ~\AppData\Roaming\Python\Python312\site-packages\pandas\core\groupby\generic.py:257, in SeriesGroupBy.aggregate(self, func, engine, engine_kwargs, *args, **kwargs)
    255 kwargs["engine"] = engine
    256 kwargs["engine_kwargs"] = engine_kwargs
--> 257 ret = self._aggregate_multiple_funcs(func, *args, **kwargs)
    258 if relabeling:
    259     # columns is not narrowed by mypy from relabeling flag
    260     assert columns is not None  # for mypy

File ~\AppData\Roaming\Python\Python312\site-packages\pandas\core\groupby\generic.py:362, in SeriesGroupBy._aggregate_multiple_funcs(self, arg, *args, **kwargs)
    360     for idx, (name, func) in enumerate(arg):
    361         key = base.OutputKey(label=name, position=idx)
--> 362         results[key] = self.aggregate(func, *args, **kwargs)
    364 if any(isinstance(x, DataFrame) for x in results.values()):
    365     from pandas import concat
...
-> 1363 raise AttributeError(
   1364     f"'{type(self).__name__}' object has no attribute '{attr}'"
   1365 )

Extracting Spatial Geometry, Temporal Cycles, and Volatility Baselines...


AttributeError: 'SeriesGroupBy' object has no attribute 'mad'

In [17]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# =====================================================================
# 🛠️ GEOCENTRIC DECODER + VOLATILITY TRACKING
# =====================================================================
def geohash_to_latlon(geohash):
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    for char in geohash:
        bit_value = base32.find(char)
        if bit_value == -1: continue
        for i in range(4, -1, -1):
            bit = (bit_value >> i) & 1
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2
                if bit == 1: lon_interval = (mid, lon_interval[1])
                else: lon_interval = (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2
                if bit == 1: lat_interval = (mid, lat_interval[1])
                else: lat_interval = (lat_interval[0], mid)
            is_even = not is_even
    return (lat_interval[0] + lat_interval[1]) / 2, (lon_interval[0] + lon_interval[1]) / 2

print("Extracting Spatial Geometry, Temporal Cycles, and Volatility Baselines...")
all_data = pd.concat([train_fe, test_fe], axis=0).reset_index(drop=True)

# Apply Spatial Decomposition
coords = [geohash_to_latlon(gh) for gh in all_data['geohash']]
all_data['latitude'] = [c[0] for c in coords]
all_data['longitude'] = [c[1] for c in coords]

# Apply Cyclic Time Encodings
all_data['sin_time'] = np.sin(2 * np.pi * all_data['time_minutes'] / 1440.0)
all_data['cos_time'] = np.cos(2 * np.pi * all_data['time_minutes'] / 1440.0)

# 🌟 FIX: Lambda approach to bypass missing .mad() attribute in Pandas GroupBy
geo_stats = all_data[all_data['is_train'] == 1].groupby('geohash')['demand'].agg([
    ('geo_demand_std', 'std'),
    ('geo_demand_mad', lambda x: np.mean(np.abs(x - np.mean(x))))
]).reset_index()

all_data = all_data.merge(geo_stats, on='geohash', how='left')
all_data['geo_demand_std'] = all_data['geo_demand_std'].fillna(all_data['demand'].std())
all_data['geo_demand_mad'] = all_data['geo_demand_mad'].fillna(all_data['demand'].std() * 0.8) # Quick approximation fallback

# Isolate back to clean matrices
train_fe_updated = all_data[all_data['is_train'] == 1].copy()
test_fe_updated = all_data[all_data['is_train'] == 0].copy()

# Deduplicate features cleanly
raw_features_v6 = [f for f in features if f != 'geohash'] + ['latitude', 'longitude', 'sin_time', 'cos_time', 'geo_demand_std', 'geo_demand_mad']
features_v6 = list(dict.fromkeys(raw_features_v6))

X_train = train_fe_updated[features_v6].values
X_test = test_fe_updated[features_v6].values
y_train = train_fe_updated['demand'].values

X_train = np.nan_to_num(X_train)
X_test = np.nan_to_num(X_test)

# =====================================================================
# 🤖 4-MODEL CROSS-VALIDATION LOOP WITH EXPANDED REBIASED HYPERPARAMETERS
# =====================================================================
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb, pred_lgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_xgb, pred_xgb = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_cat, pred_cat = np.zeros(len(X_train)), np.zeros(len(X_test))
oof_et,  pred_et  = np.zeros(len(X_train)), np.zeros(len(X_test))

y_train_log = np.log1p(y_train)

print("="*60)
print("TRAINING HETEROGENEOUS ENSEMBLE WITH VOLATILITY TRACKING")
print("="*60)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr_log, y_val_log = y_train_log[tr_idx], y_train_log[val_idx]
    y_val_true = y_train[val_idx]
    
    # 1. LightGBM 
    lgb_params = {
        'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
        'learning_rate': 0.025, 'num_leaves': 91, 'max_depth': -1,
        'min_child_samples': 25, 'feature_fraction': 0.75, 'bagging_fraction': 0.8,
        'bagging_freq': 5, 'verbose': -1, 'n_jobs': -1, 'random_state': 42
    }
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr_log)
    dval_lgb = lgb.Dataset(X_val, label=y_val_log)
    model_lgb = lgb.train(lgb_params, dtrain_lgb, num_boost_round=3500, valid_sets=[dval_lgb], callbacks=[lgb.early_stopping(120, verbose=False)])
    oof_lgb[val_idx] = np.expm1(model_lgb.predict(X_val))
    pred_lgb += np.expm1(model_lgb.predict(X_test)) / N_FOLDS
    
    # 2. XGBoost
    xgb_params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'learning_rate': 0.025,
        'max_depth': 7, 'min_child_weight': 12, 'subsample': 0.8, 'colsample_bytree': 0.75,
        'tree_method': 'hist', 'random_state': 42
    }
    dtrain_xgb = xgb.DMatrix(X_tr, label=y_tr_log, feature_names=features_v6)
    dval_xgb = xgb.DMatrix(X_val, label=y_val_log, feature_names=features_v6)
    dtest_xgb = xgb.DMatrix(X_test, feature_names=features_v6)
    model_xgb = xgb.train(xgb_params, dtrain_xgb, num_boost_round=3500, evals=[(dval_xgb, 'val')], early_stopping_rounds=120, verbose_eval=False)
    oof_xgb[val_idx] = np.expm1(model_xgb.predict(dval_xgb))
    pred_xgb += np.expm1(model_xgb.predict(dtest_xgb)) / N_FOLDS
    
    # 3. CatBoost
    model_cat = CatBoostRegressor(iterations=3500, learning_rate=0.025, depth=8, random_seed=42, verbose=0, early_stopping_rounds=120, eval_metric='RMSE')
    model_cat.fit(X_tr, y_tr_log, eval_set=(X_val, y_val_log), verbose=0)
    oof_cat[val_idx] = np.expm1(model_cat.predict(X_val))
    pred_cat += np.expm1(model_cat.predict(X_test)) / N_FOLDS
    
    # 4. ExtraTrees (Deepened spatial-variance engine)
    model_et = ExtraTreesRegressor(n_estimators=200, max_depth=25, min_samples_split=4, max_features=0.6, n_jobs=-1, random_state=42)
    model_et.fit(X_tr, y_train[tr_idx])
    oof_et[val_idx] = model_et.predict(X_val)
    pred_et += model_et.predict(X_test) / N_FOLDS
    
    fold_blend = (oof_lgb[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx] + oof_et[val_idx]) / 4.0
    print(f"Fold {fold+1} Volatility Blend R2: {r2_score(y_val_true, fold_blend):.6f}")

# =====================================================================
# 🎛️ MATHEMATICAL WEIGHT OPTIMIZATION
# =====================================================================
print("\n" + "="*60)
print("OPTIMIZING BLEND MATRIX WEIGHTS (SLSQP)")
print("="*60)

def r2_objective(weights):
    blend = weights[0]*oof_lgb + weights[1]*oof_xgb + weights[2]*oof_cat + weights[3]*oof_et
    return -r2_score(y_train, blend)

cons = ({'type': 'eq', 'fun': lambda w: 1.0 - np.sum(w)})
bounds = [(0, 1)] * 4
opt_res = minimize(r2_objective, [0.1, 0.2, 0.2, 0.5], method='SLSQP', bounds=bounds, constraints=cons)
best_w = opt_res.x

print(f"Optimal Weights -> LGB: {best_w[0]:.4f} | XGB: {best_w[1]:.4f} | CAT: {best_w[2]:.4f} | ET: {best_w[3]:.4f}")
print(f"Individual OOF R2 Scores:")
print(f"  LightGBM R2:   {r2_score(y_train, oof_lgb):.6f}")
print(f"  XGBoost R2:    {r2_score(y_train, oof_xgb):.6f}")
print(f"  CatBoost R2:   {r2_score(y_train, oof_cat):.6f}")
print(f"  ExtraTrees R2: {r2_score(y_train, oof_et):.6f}")
print(f"👉 MAXIMIZED VOLATILITY ENSEMBLE OOF R2: {-opt_res.fun:.6f}\n")

# Output Processing
final_pred = best_w[0]*pred_lgb + best_w[1]*pred_xgb + best_w[2]*pred_cat + best_w[3]*pred_et
final_pred = np.clip(final_pred, 0, None)

submission = pd.DataFrame({'Index': test_idx, 'demand': final_pred})
submission.to_csv('./predicted_demand_volatility.csv', index=False)
print(f"Saved: predicted_demand_volatility.csv ({submission.shape[0]} rows)")

Extracting Spatial Geometry, Temporal Cycles, and Volatility Baselines...
TRAINING HETEROGENEOUS ENSEMBLE WITH VOLATILITY TRACKING
Fold 1 Volatility Blend R2: 0.960439
Fold 2 Volatility Blend R2: 0.960574
Fold 3 Volatility Blend R2: 0.961430
Fold 4 Volatility Blend R2: 0.956854
Fold 5 Volatility Blend R2: 0.959952

OPTIMIZING BLEND MATRIX WEIGHTS (SLSQP)
Optimal Weights -> LGB: 0.0976 | XGB: 0.1539 | CAT: 0.1032 | ET: 0.6453
Individual OOF R2 Scores:
  LightGBM R2:   0.957568
  XGBoost R2:    0.957215
  CatBoost R2:   0.956903
  ExtraTrees R2: 0.959950
👉 MAXIMIZED VOLATILITY ENSEMBLE OOF R2: 0.960747

Saved: predicted_demand_volatility.csv (41778 rows)


## 3. Model Training (5-Fold CV)

In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

### 3a. LightGBM

In [ ]:
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42,
}

oof_lgb = np.zeros(len(X_train))
pred_lgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val)
    
    model = lgb.train(lgb_params, dtrain, num_boost_round=3000,
                      valid_sets=[dval],
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    
    oof_lgb[val_idx] = model.predict(X_val)
    pred_lgb += model.predict(X_test) / N_FOLDS
    print(f'Fold {fold+1}: R2 = {r2_score(y_val, oof_lgb[val_idx]):.6f}')

print(f'\nLightGBM OOF R2: {r2_score(y_train, oof_lgb):.6f}')

### 3b. XGBoost

In [ ]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.03,
    'max_depth': 8,
    'min_child_weight': 10,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'tree_method': 'hist',
    'random_state': 42,
}

oof_xgb = np.zeros(len(X_train))
pred_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=features)
    dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
    dtest = xgb.DMatrix(X_test, feature_names=features)
    
    model = xgb.train(xgb_params, dtrain, num_boost_round=3000,
                      evals=[(dval, 'val')], early_stopping_rounds=100, verbose_eval=0)
    
    oof_xgb[val_idx] = model.predict(dval)
    pred_xgb += model.predict(dtest) / N_FOLDS
    print(f'Fold {fold+1}: R2 = {r2_score(y_val, oof_xgb[val_idx]):.6f}')

print(f'\nXGBoost OOF R2: {r2_score(y_train, oof_xgb):.6f}')

configurations made by rajkumar upto now:

🗺️ Problem ContextThe task is a high-stakes spatial-temporal tabular regression problem predicting location-based demand. The data features an anonymized categorical geohash representing location zones, alongside structured timeline parameters (day, time_minutes).🔄 The Evolutionary Lifecycle of Our ModelsIteration 1: The Standard GBDT Stack (Baseline)Approach: Built an ensemble using LightGBM, XGBoost, and CatBoost wrapped in a GroupKFold spatial holdout split based directly on raw geohash strings. Target variance was stabilized using a log-transformation (np.log1p). Final predictions were blended mathematically using Scipy's SLSQP optimizer.Result: Local OOF $R^2 \approx 95.6\%$, Public Test Score: 91.066Takeaway: A strong baseline, but the tree-based models were hitting an architectural ceiling because they all computed errors sequentially, leading them to overfit identical spatial anomalies.Iteration 2: The Time-Series "Lag" Trap (Failure)Approach: Attempted to introduce standard time-series engineering by building 24-hour historical lags and drift trends mapped across a single-fold temporal split (Training on Day 48, Validating on Day 49). A linear stabilizer (Ridge Regression) was thrown into the blend matrix.Result: Dropped to 90.68Why it failed: 1. Truncating the dataset to a rigid time-split threw away over 50% of the training data.2. Tree architectures struggle to generalize absolute timeline shifts when spatial distributions vary radically day-to-day, introducing massive multicollinearity and validation leaks.Iteration 3: Geocentric Geometry & Heterogeneous Anchoring (The Breakthrough)Approach: We completely overhauled how the models perceived physical space and time, shifting away from standard GBDTs to a heterogeneous architecture:Spatial Decomposition: Decoded base32 geohash strings into absolute continuous geographical float center points (latitude and longitude). This allowed models to build exact spatial bounding boxes rather than guessing string classifications.Cyclic Time Encodings: Converted raw time_minutes into continuous cyclical Sine ($\sin$) and Cosine ($\cos$) waves, forcing the models to realize that minute 0 (12:00 AM) and minute 1439 (11:59 PM) are temporally adjacent.Ensemble Diversification (ExtraTrees): Added an ExtraTrees Regressor (Extremely Randomized Trees) as a 4th model trained directly on the raw scale.Result: Beat the target score, jumping to 91.1975Why it succeeded: Look at the Scipy optimization output from this run:PlaintextOptimal Weights -> LGB: 0.1080 | XGB: 0.1818 | CAT: 0.2290 | ET: 0.4812
The optimizer assigned a massive 48.12% weight to ExtraTrees. Because GBDTs construct splits sequentially to aggressively drive down training error, they are prone to capturing noise. ExtraTrees splits nodes completely at random, radically reducing prediction variance and serving as a structural anchor for the test set.Iteration 4: Spatial Volatility Tracking & Deep Tuning (Current)Approach: We added Spatial Demand Volatility metrics—specifically calculating the Standard Deviation (std) and Mean Absolute Deviation (mad via custom lambda aggregation to support modern Pandas builds) of past demand per geohash. This actively teaches the models which coordinates are inherently volatile vs. completely stable. ExtraTrees was further deep-tuned (max_depth=25, n_estimators=200) to maximize its spatial routing capacity.🧠 Core Engineering Principles LearnedVary Your Model Architectures: Blending three different types of gradient-boosted trees doesn't stop overfitting if they all think the same way. Introducing a completely different family of models (like a random-forest variant or ExtraTrees) reduces structural bias.Make Geometry Continuous: Tree algorithms struggle immensely with high-cardinality string categories like Geohashes. Decoding them into lat/lon floats unlocks massive performance gains.Log Scale Is Mandatory for Demand: When managing geographical data with massive density spikes (e.g., city centers vs. suburbs), optimizing GBDTs on np.log1p(y) prevents high-volume zones from completely drowning out the gradients of smaller zones.


### 3c. CatBoost

In [ ]:
oof_cat = np.zeros(len(X_train))
pred_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    model = CatBoostRegressor(
        iterations=3000, learning_rate=0.03, depth=8,
        l2_leaf_reg=3, random_seed=42, verbose=0,
        early_stopping_rounds=100, eval_metric='RMSE',
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=0)
    
    oof_cat[val_idx] = model.predict(X_val)
    pred_cat += model.predict(X_test) / N_FOLDS
    print(f'Fold {fold+1}: R2 = {r2_score(y_val, oof_cat[val_idx]):.6f}')

print(f'\nCatBoost OOF R2: {r2_score(y_train, oof_cat):.6f}')

## 4. Optimal Ensemble Blending

In [ ]:
best_r2 = -999
best_w = (0, 0, 0)

for w1 in np.arange(0.1, 0.9, 0.05):
    for w2 in np.arange(0.1, 0.9 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
        r2 = r2_score(y_train, blend)
        if r2 > best_r2:
            best_r2 = r2
            best_w = (w1, w2, w3)

print(f'Optimal Weights -> LGB: {best_w[0]:.2f}, XGB: {best_w[1]:.2f}, CAT: {best_w[2]:.2f}')
print(f'\nIndividual R2 scores:')
print(f'  LightGBM:  {r2_score(y_train, oof_lgb):.6f}')
print(f'  XGBoost:   {r2_score(y_train, oof_xgb):.6f}')
print(f'  CatBoost:  {r2_score(y_train, oof_cat):.6f}')
print(f'  Ensemble:  {best_r2:.6f}')

## 5. Save Predictions

In [ ]:
final_pred = best_w[0] * pred_lgb + best_w[1] * pred_xgb + best_w[2] * pred_cat

submission = pd.DataFrame({'Index': test_idx, 'demand': final_pred})
submission.to_csv('./predicted_demand.csv', index=False)

print(f'Saved: predicted_demand.csv ({submission.shape[0]} rows)')
submission.head(10)

## 6. Save Approach Document

In [ ]:
approach_text = f"""================================================================================
                    DEMAND PREDICTION - APPROACH DOCUMENT
================================================================================

1. PROBLEM STATEMENT
--------------------
Predict the 'demand' (continuous float) for various geographic locations (geohash)
at specific timestamps, given road characteristics, weather, and temperature data.
This is a regression task, optimized for maximum R2 score.

Dataset: 77,299 train rows / 41,778 test rows / 11 columns


2. DATA OVERVIEW
----------------
Columns:
  - Index          : Row identifier (int)
  - geohash        : Geographic hash code (1,249 unique - HIGH cardinality)
  - day            : Day number (48 or 49)
  - timestamp      : Time in "H:M" format (96 unique, 15-min intervals)
  - demand         : TARGET variable (continuous float)
  - RoadType       : Categorical - Residential/Street/Highway (600 missing)
  - NumberofLanes  : Integer (1-5)
  - LargeVehicles  : Binary - Allowed/Not Allowed
  - Landmarks      : Binary - Yes/No
  - Temperature    : Float, degrees (2,495 missing)
  - Weather        : Categorical - Sunny/Rainy/Foggy/Snowy (797 missing)


3. FEATURE ENGINEERING
----------------------
a) Timestamp Decomposition:
   - Extracted 'hour' and 'minute' from "H:M" format
   - Created 'time_minutes' = hour*60 + minute
   - Cyclical sin/cos encoding for hour and minutes
   - Time bucket: binned hours into 6 periods

b) Geohash Engineering (High Cardinality - 1249 unique):
   - Label encoded the full geohash for tree models
   - Extracted geohash prefixes at 4-char and 5-char levels
   - Target encoding: mean, std, and count of demand per geohash

c) Categorical Encoding:
   - RoadType: mapped to 0/1/2 (NaN left as-is for GBDT)
   - Weather: mapped to 0/1/2/3 (NaN left as-is for GBDT)
   - LargeVehicles & Landmarks: binary encoded

d) Missing Value Strategy:
   - GBDT models handle NaN natively with optimal split direction
   - Added 'Temperature_missing' binary indicator

e) Interaction Features:
   - lanes_x_road, temp_x_weather, lanes_x_landmarks, lanes_x_largeveh

Total features: {len(features)}




6. TOOLS & LIBRARIES USED
--------------------------
- Python 3.13
- pandas: data loading & manipulation
- numpy: numerical operations
- scikit-learn: KFold CV, R2 metric, LabelEncoder
- LightGBM: gradient boosting model
- XGBoost: gradient boosting model
- CatBoost: gradient boosting model


7. SOURCE FILES
---------------
- dataset/train.csv          : Training data (77,299 rows x 11 columns)
- dataset/test.csv           : Test data (41,778 rows x 10 columns)
- dataset/sample_submission.csv : Submission format
- eda.ipynb                  : Exploratory data analysis notebook
- pipeline.ipynb             : Full training & prediction pipeline
- predicted_demand.csv       : Final predictions output
- approach.txt               : This approach document
================================================================================
"""

with open('./approach.txt', 'w') as f:
    f.write(approach_text)

print('Saved: approach.txt')
print('\nDONE! All files saved successfully.')